In [1]:
import mteb
import pandas as pd
import os
from sentence_transformers import SentenceTransformer
import torch
from dotenv import load_dotenv

load_dotenv()

/home/sagemaker-user/bertimbau-sentence-transformer/.venv_st/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.mps.is_available() else "cpu"
DEVICE

'cuda'

In [3]:
tasks = mteb.get_tasks(languages=["por"], modalities=['text'])

for task in tasks:
    print(task)

BibleNLPBitextMining(name='BibleNLPBitextMining', languages=['eng', 'por'])
FloresBitextMining(name='FloresBitextMining', languages=['ace', 'acm', 'acq', '...'])
NTREXBitextMining(name='NTREXBitextMining', languages=['arb', 'ben', 'cat', '...'])
TatoebaBitextMining(name='Tatoeba', languages=['eng', 'por'])
WebFAQBitextMiningQAs(name='WebFAQBitextMiningQAs', languages=['cat', 'dan', 'deu', '...'])
WebFAQBitextMiningQuestions(name='WebFAQBitextMiningQuestions', languages=['cat', 'dan', 'deu', '...'])
AfriSentiClassification(name='AfriSentiClassification', languages=['por'])
AfriSentiLangClassification(name='AfriSentiLangClassification', languages=['amh', 'arq', 'ary', '...'])
LanguageClassification(name='LanguageClassification', languages=['ara', 'bul', 'cmn', '...'])
MassiveIntentClassification(name='MassiveIntentClassification', languages=['por'])
MassiveScenarioClassification(name='MassiveScenarioClassification', languages=['por'])
MultiHateClassification(name='MultiHateClassification

In [4]:
# model_name = "neuralmind/bert-base-portuguese-cased"
model_name = "./legal-quati-10M/checkpoint-80"
tasks = [
    ("Assin2STS", None),
    ("SICK-BR-STS", None),
    ("STSBenchmarkMultilingualSTS", 'pt'),
    
    ('MassiveIntentClassification', 'pt'),
    ('MultiHateClassification', 'por'),
    ('BrazilianToxicTweetsClassification', None),
    ('HateSpeechPortugueseClassification', None),
    ('TweetSentimentClassification', 'portuguese'),

    ('MultiLongDocReranking', 'pt'),
    ('WikipediaRerankingMultilingual', 'pt'),
    ('XGlueWPRReranking', 'pt'),

    ('WebFAQRetrieval', 'por'),
    ('MultiLongDocRetrieval', 'pt'),
    ('WikipediaRetrievalMultilingual', 'pt')
]

In [5]:
# model_meta = mteb.get_model(model_name, device="cuda")
model_meta = SentenceTransformer(model_name, device=DEVICE)

# select the desired tasks and evaluate
task_name_list = []
model_name_list = []
main_score_list = []

for task_info in tasks:
    print(f"""
    #############################

    Avaliando {task_info[0]} ({task_info[1]})...
    
    #############################
    """)

    task = mteb.get_task(task_info[0], languages=['por'], hf_subsets=task_info[1])

    # with encode kwargs
    result = mteb.evaluate(model_meta, task, encode_kwargs={"batch_size": 256})

    task_name = result.task_results[0].task_name
    model_name = result.model_name
    main_score = result.task_results[0].main_score

    task_name_list.append(task_name)
    model_name_list.append(model_name)
    main_score_list.append(main_score)

    print(f"Main Score: {main_score}")

    del task, result
    torch.cuda.empty_cache()

print("Avaliação concluída!")

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3514.81it/s]



    #############################

    Avaliando Assin2STS (None)...

    #############################
    
Main Score: 0.813186

    #############################

    Avaliando SICK-BR-STS (None)...

    #############################
    
Main Score: 0.912578

    #############################

    Avaliando STSBenchmarkMultilingualSTS (pt)...

    #############################
    
Main Score: 0.8583540000000001

    #############################

    Avaliando MassiveIntentClassification (pt)...

    #############################
    


KeyboardInterrupt: 

In [ ]:
df_results = pd.DataFrame({
    'model_name': model_name_list,
    'task_name': task_name_list,
    'main_score': main_score_list
})

In [ ]:
filepath = '../data/results_eval_mteb.csv'

if os.path.exists(filepath):
    df_results_cache = pd.read_csv(filepath)
    df_results = pd.concat([df_results_cache, df_results], axis=0, ignore_index=True)

df_results.to_csv(filepath, index=False)